In [ ]:
# Cell 1
# Mount Drive and set workspace
import os
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

def find_baseline_dir():
    candidates = [
        "/content/drive/MyDrive/final_project/baseline",
        "/content/drive/MyDrive/final_project/baseline/",
    ]

    for p in candidates:
        if os.path.isdir(p):
            return os.path.abspath(p)

    shared_root = "/content/drive/Shareddrives"
    if os.path.isdir(shared_root):
        for root, dirs, _ in os.walk(shared_root):
            if root.endswith("/final_project") and "baseline" in dirs:
                return os.path.abspath(os.path.join(root, "baseline"))

    raise FileNotFoundError("Could not find final_project/baseline in Drive.")

BASE_DIR = find_baseline_dir()
BASE_DIR = os.path.realpath(BASE_DIR)

os.chdir(BASE_DIR)

print("BASE_DIR =", BASE_DIR)
print("CWD =", os.getcwd())

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Mounted at /content/drive
BASE_DIR = /content/drive/.shortcut-targets-by-id/1V7smEWLD_ZhlaD773UjRiHThZZ9cpgS-/final_project/baseline
CWD = /content/drive/.shortcut-targets-by-id/1V7smEWLD_ZhlaD773UjRiHThZZ9cpgS-/final_project/baseline
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


In [ ]:
# Cell 2
# Clone repo without auto-pull
import os
import subprocess

REPO_URL = "https://github.com/ali-mohmmadi/KGP-CuriousLLM.git"
REPO_DIR = "/content/KGP-CuriousLLM"

REPO_COMMIT = None  # Set a commit hash here only if you know the original run commit.

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Cloning repository into:", REPO_DIR)
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
else:
    print("Repo already exists. Skipping git pull to keep code stable.")

if REPO_COMMIT:
    subprocess.check_call(["git", "-C", REPO_DIR, "checkout", REPO_COMMIT])

current_commit = subprocess.check_output(
    ["git", "-C", REPO_DIR, "rev-parse", "HEAD"],
    text=True,
).strip()

print("Repo ready at:", REPO_DIR)
print("Repo commit:", current_commit)
print("Repo root files:", os.listdir(REPO_DIR)[:15])

Repo already exists. Skipping git pull to keep code stable.
Repo ready at: /content/KGP-CuriousLLM
Repo commit: 7d7e864f7cd0a06817a43a40fe333ea399ed403e
Repo root files: ['create_dirs.py', 'ft_mistral_main.py', 'KGP', 'configs', '.gitignore', 'MDR_main.py', 'kgp_main.py', 'requirements.txt', 'images', 'kg_construct_main.py', 'MDR_embedding_main.py', 'quantize_mistral_main.py', '.git', 'T5_main.py', 'grid_search_mistral_main.py']


In [ ]:
# Cell 3
# Install dependencies
import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "-U", "pip", "setuptools", "wheel", "-q"
])

pkgs = [
    "pyyaml==6.0.1",
    "tqdm==4.66.2",
    "numpy==1.26.4",
    "pandas==2.2.2",
    "networkx==3.3",
    "scikit-learn==1.4.2",
    "rank_bm25",
    "nltk",
    "sentence-transformers",
    "transformers==5.0.0",
    "accelerate==1.13.0",
    "peft==0.19.1",
    "bitsandbytes",
    "sentencepiece",
    "huggingface-hub>=0.28.1",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

import yaml, tqdm, numpy, pandas, networkx, sklearn, transformers, accelerate, peft

print("pyyaml:", yaml.__version__)
print("numpy:", numpy.__version__)
print("pandas:", pandas.__version__)
print("networkx:", networkx.__version__)
print("scikit-learn:", sklearn.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("peft:", peft.__version__)
print("Installed OK")

pyyaml: 6.0.1
numpy: 1.26.4
pandas: 2.2.2
networkx: 3.3
scikit-learn: 1.4.2
transformers: 5.0.0
accelerate: 1.13.0
peft: 0.19.1
Installed OK


In [ ]:
# Cell 4
# Add repo to path
import os
import sys

if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("PYTHONPATH ready")
print("BASE_DIR in path:", BASE_DIR in sys.path)
print("REPO_DIR in path:", REPO_DIR in sys.path)

PYTHONPATH ready
BASE_DIR in path: True
REPO_DIR in path: True


In [ ]:
# Cell 5
# Check required files
import os
import json
import numpy as np

RUN_ID = "hotpotqa_dev2017wiki_1000_old"

DATASET_PATH = os.path.join(BASE_DIR, "DATA", "HotpotQA", "hotpotqa_dev_2017wiki_1000_converted.json")
GRAPH_PATH = os.path.join(BASE_DIR, "DATA", "KG", "graphs", f"graph_{RUN_ID}", "graph.gpickle")

for p in [DATASET_PATH, GRAPH_PATH]:
    assert os.path.isfile(p), f"Missing file: {p}"

data = json.load(open(DATASET_PATH, "r", encoding="utf-8"))

print("DATASET_PATH =", DATASET_PATH)
print("GRAPH_PATH   =", GRAPH_PATH)
print("num_questions =", len(data))
print("example_keys =", list(data[0].keys()))
print("type_counts preview:")
from collections import Counter
print(Counter(x.get("type", "unknown") for x in data))

DATASET_PATH = /content/drive/.shortcut-targets-by-id/1V7smEWLD_ZhlaD773UjRiHThZZ9cpgS-/final_project/baseline/DATA/HotpotQA/hotpotqa_dev_2017wiki_1000_converted.json
GRAPH_PATH   = /content/drive/.shortcut-targets-by-id/1V7smEWLD_ZhlaD773UjRiHThZZ9cpgS-/final_project/baseline/DATA/KG/graphs/graph_hotpotqa_dev2017wiki_1000_old/graph.gpickle
num_questions = 1000
example_keys = ['question', 'answer', 'type', 'titles', 'docs_chunks', 'docs', 'title_chunks', 'supports']
type_counts preview:
Counter({'bridge': 700, 'comparison': 300})


In [ ]:
# Cell 6
# Write resume config
import os
import yaml
import torch

cfg_dir = os.path.join(BASE_DIR, "configs", "kgp")
os.makedirs(cfg_dir, exist_ok=True)

CFG_PATH = os.path.join(cfg_dir, "qwen_agent_hotpot_1000.yml")

device = "cuda" if torch.cuda.is_available() else "cpu"

RUN_ID = "hotpotqa_dev2017wiki_1000_old"

args_dict = {
    "root_dir": BASE_DIR,
    "seed": 2048,
    "dataset": "hotpot",
    "question_dataset": "DATA/HotpotQA/hotpotqa_dev_2017wiki_1000_converted.json",
    "device": device,
    "init_retriever": {
        "name": "tfidf",
        "no_traversal": False,
        "no_traversal_topk": 30,
        "topk": 4,
    },
    "retriever": {
        "name": "qwen",
        "model": "Qwen/Qwen3-8B",
        "adapter": "mohammad-shirkhani/Qwen3_finetune_follow_up_question",
        "load_in_4bit": True,
        "model_params": {
            "lora_rank": 32,
            "lora_layers": 32,
        },
        "inference_params": {
            "temp": 0.6,
            "top_p": 0.85,
            "max_token_len": 50,
        },
        "T5_params": {
            "model_path": "models/checkpoints_t5/reason_t5-large",
            "max_source_length": 512,
            "max_target_length": 512,
        },
        "traversal_params": {
            "n_hop": 2,
            "n_neighbors": 3,
        },
    },
    "KG": f"DATA/KG/graphs/graph_{RUN_ID}/graph.gpickle",
    "emb_model": {
        "model": "sentence-transformers/multi-qa-MiniLM-L6-cos-v1",
    },
    "checkpoint": {
        "id": "qwen_agent",
        "resume": True,
        "save_every": 1,
        "save_dir": "DATA/KG/evidence/hotpot_evidence_1000",
        "checkpoint_path": "DATA/KG/evidence/hotpot_evidence_1000/qwen_agent/cp_evidence.json",
    },
}

with open(CFG_PATH, "w", encoding="utf-8") as f:
    yaml.dump(args_dict, f, sort_keys=False)

print("Wrote config:", CFG_PATH)
print(yaml.safe_load(open(CFG_PATH, "r", encoding="utf-8")))

Wrote config: /content/drive/.shortcut-targets-by-id/1V7smEWLD_ZhlaD773UjRiHThZZ9cpgS-/final_project/baseline/configs/kgp/qwen_agent_hotpot_1000.yml
{'root_dir': '/content/drive/.shortcut-targets-by-id/1V7smEWLD_ZhlaD773UjRiHThZZ9cpgS-/final_project/baseline', 'seed': 2048, 'dataset': 'hotpot', 'question_dataset': 'DATA/HotpotQA/hotpotqa_dev_2017wiki_1000_converted.json', 'device': 'cuda', 'init_retriever': {'name': 'tfidf', 'no_traversal': False, 'no_traversal_topk': 30, 'topk': 4}, 'retriever': {'name': 'qwen', 'model': 'Qwen/Qwen3-8B', 'adapter': 'mohammad-shirkhani/Qwen3_finetune_follow_up_question', 'load_in_4bit': True, 'model_params': {'lora_rank': 32, 'lora_layers': 32}, 'inference_params': {'temp': 0.6, 'top_p': 0.85, 'max_token_len': 50}, 'T5_params': {'model_path': 'models/checkpoints_t5/reason_t5-large', 'max_source_length': 512, 'max_target_length': 512}, 'traversal_params': {'n_hop': 2, 'n_neighbors': 3}}, 'KG': 'DATA/KG/graphs/graph_hotpotqa_dev2017wiki_1000_old/graph.gp

In [ ]:
# Cell 6A
# Check existing checkpoint before loading models
import os
import json
import glob
import shutil
import time

DRIVE_SAVE_DIR = os.path.join(
    BASE_DIR,
    "DATA",
    "KG",
    "evidence",
    "hotpot_evidence_1000",
    "qwen_agent",
)

DRIVE_CP_PATH = os.path.join(DRIVE_SAVE_DIR, "cp_evidence.json")
DRIVE_FINAL_PATH = os.path.join(DRIVE_SAVE_DIR, "evidence.json")

def try_load_records(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            records = json.load(f)
        if not isinstance(records, list):
            return None, "not a list"
        return records, None
    except Exception as e:
        return None, str(e)

candidates = []

for p in [DRIVE_CP_PATH, DRIVE_FINAL_PATH]:
    if os.path.isfile(p):
        candidates.append(p)

backup_patterns = [
    os.path.join(DRIVE_SAVE_DIR, "cp_evidence*.json"),
    os.path.join(DRIVE_SAVE_DIR, "*.bak*.json"),
]

for pattern in backup_patterns:
    for p in glob.glob(pattern):
        if p not in candidates:
            candidates.append(p)

valid_candidates = []

for p in candidates:
    records, err = try_load_records(p)
    if records is not None:
        valid_candidates.append((len(records), p))
        print("Valid checkpoint:", len(records), p)
    else:
        print("Invalid checkpoint:", p, "|", err)

if not valid_candidates:
    raise RuntimeError(
        "No valid checkpoint found. Do not start Cell 12. "
        "Recover cp_evidence.json from Google Drive version history first."
    )

valid_candidates.sort(reverse=True)
BEST_CP_LEN, BEST_CP_SOURCE = valid_candidates[0]

print("Best checkpoint source:", BEST_CP_SOURCE)
print("Best checkpoint records:", BEST_CP_LEN)

if BEST_CP_LEN == 0:
    raise RuntimeError("Checkpoint has 0 records. Stopping to avoid starting from scratch.")

timestamp = time.strftime("%Y%m%d_%H%M%S")
backup_path = os.path.join(DRIVE_SAVE_DIR, f"cp_evidence_before_resume_{timestamp}.json")

if os.path.abspath(BEST_CP_SOURCE) == os.path.abspath(DRIVE_CP_PATH):
    shutil.copy2(DRIVE_CP_PATH, backup_path)
    print("Backup created:", backup_path)
else:
    print("Best checkpoint is not the main cp file. No main cp backup made.")

Valid checkpoint: 545 /content/drive/.shortcut-targets-by-id/1V7smEWLD_ZhlaD773UjRiHThZZ9cpgS-/final_project/baseline/DATA/KG/evidence/hotpot_evidence_1000/qwen_agent/cp_evidence.json
Best checkpoint source: /content/drive/.shortcut-targets-by-id/1V7smEWLD_ZhlaD773UjRiHThZZ9cpgS-/final_project/baseline/DATA/KG/evidence/hotpot_evidence_1000/qwen_agent/cp_evidence.json
Best checkpoint records: 545
Backup created: /content/drive/.shortcut-targets-by-id/1V7smEWLD_ZhlaD773UjRiHThZZ9cpgS-/final_project/baseline/DATA/KG/evidence/hotpot_evidence_1000/qwen_agent/cp_evidence_before_resume_20260506_070343.json


In [ ]:
#Cell 7
# Import repo-faithful core functions and local deps
import os
import re
import json
import yaml
import pickle
import random
import numpy as np
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

from KGP.Traversal_agents.KGP import get_supporting_evidence

print("Core imports OK")

Core imports OK


In [ ]:
#Cell 8
# Define local helpers and a Colab-safe TF-IDF seed retriever
def load_graph(graph_path):
    with open(graph_path, "rb") as f:
        return pickle.load(f)

def generate_evidence_record(q_type, question, evidence, answer, supports):
    return {
        "type": q_type,
        "question": question,
        "evidence": evidence,
        "answer": answer,
        "supports": supports,
    }

def clean_followup(text):
    text = text.strip()
    text = text.replace("<|im_end|>", " ").replace("<|endoftext|>", " ").strip()

    if "Follow-up Question:" in text:
        text = text.split("Follow-up Question:")[-1].strip()

    if "Question:" in text and len(text.splitlines()) == 1:
        text = text.split("Question:")[-1].strip()

    text = re.sub(r"^\d+\s*", "", text).strip()
    text = text.strip().strip('"').strip("'").strip()
    text = text.split("\n")[0].strip()

    return text if text else "NA"

def get_titled_docs_from_kg(G, doc_field="passage", title_field="title"):
    nodes = list(G.nodes)
    nodes.sort()

    documents = []
    for node in nodes:
        passage = G.nodes[node][doc_field]
        title = G.nodes[node][title_field]
        combined_doc = "TITLE: " + title + "." + " " + passage
        documents.append(combined_doc)

    return documents

class TF_IDF_Retriever:
    def __init__(self, topk, G):
        self.topk = topk
        self.G = G
        self.all_documents = get_titled_docs_from_kg(self.G)
        self.vectorizer, self.tfidf_matrix = self.init_model(self.all_documents)

    def retrieve(self, query):
        query_emb = self.vectorizer.transform([query])
        cosine_sim = cosine_similarity(query_emb, self.tfidf_matrix).flatten()
        return cosine_sim.argsort()[-self.topk:][::-1]

    @staticmethod
    def init_model(all_documents):
        vectorizer = TfidfVectorizer()
        tfidf_matrix = vectorizer.fit_transform(all_documents)
        return vectorizer, tfidf_matrix

class GoldStandard_Retriever:
    def __init__(self, topk, dataset):
        self.topk = topk
        self.gold_standard = {i["question"]: i["supports"] for i in dataset}

    def retrieve(self, query):
        supports = self.gold_standard[query]
        return [i[1] for i in supports]

class No_Retriever:
    def __init__(self, topk, G=None):
        self.topk = topk
        self.G = G
        self.all_documents = get_titled_docs_from_kg(G) if G is not None else []

    def retrieve(self, query):
        return []

def get_seeding_retriever_local(name, topk, G=None, dataset=None):
    if name == "tfidf":
        return TF_IDF_Retriever(topk=topk, G=G)
    elif name == "gold":
        return GoldStandard_Retriever(topk=topk, dataset=dataset)
    elif name == "none":
        return No_Retriever(topk=topk, G=G)
    else:
        raise ValueError(f"Retriever {name} not implemented in this Colab notebook.")

In [ ]:
# Cell 9
# Define Qwen inference and Qwen agent
class Qwen_Inference:
    def __init__(self, model, tokenizer, temp=1.0, top_p=1.0, max_token_len=100,
                 parse_template=True):
        self.model = model
        self.tokenizer = tokenizer
        self.temp = temp
        self.top_p = top_p
        self.max_token_len = max_token_len
        self.parse_template = parse_template

    def get_question(self, prompt, verbose=0):
        if self.parse_template:
            instruction = """You are a critical thinker and like to ask questions.
Please provide only the follow-up question without additional information.
Please think carefully and provide a follow-up question that is relevant to the previous conversation."""
            prompt_messages = [
                {"role": "user", "content": instruction},
                {"role": "assistant", "content": prompt},
            ]
            prompt_text = self.tokenizer.apply_chat_template(
                prompt_messages,
                tokenize=False,
                add_generation_prompt=False,
            )
        else:
            prompt_text = prompt

        device = next(self.model.parameters()).device
        inputs = self.tokenizer(
            prompt_text,
            return_tensors="pt",
            truncation=True,
            max_length=2048,
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.inference_mode():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.max_token_len,
                do_sample=(self.temp > 0),
                temperature=max(self.temp, 1e-5),
                top_p=self.top_p,
                repetition_penalty=1.05,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )

        gen_ids = outputs[0][inputs["input_ids"].shape[1]:]
        text = self.tokenizer.decode(gen_ids, skip_special_tokens=True)
        text = clean_followup(text)

        if verbose:
            print(text)

        return text


class Qwen_Agent:
    def __init__(self, args):
        self.args = args
        self.retriever = self.init_qwen_inference(args)
        self.emb = self.init_sentence_transformer(args)

    def get_top_k_neighbors(self, G, curr_path, prompt):
        new_prompt = self.get_prompt(prompt, curr_path, G)
        question = self.retriever.get_question(new_prompt)

        # Keep repo logic
        if "NA" in question and len(question) < 30:
            return []

        if "Question:" in question:
            question = "".join(question.split("Question:")[1:]).strip()

        neighbors = list(G.neighbors(curr_path[-1]))
        if len(neighbors) == 0:
            return []

        question_emb = self.emb.encode(question, device=self.args["device"])

        neighbors_passages = [
            f"Title: {G.nodes[neighbor]['title']}. {G.nodes[neighbor]['passage']}"
            for neighbor in neighbors
        ]

        neighbors_emb = self.emb.encode(neighbors_passages, device=self.args["device"])
        sim_scores = util.dot_score(question_emb, neighbors_emb).cpu().numpy().flatten()

        # Keep repo ordering style
        top_neighbors_indices = sim_scores.argsort()[
            -self.args["retriever"]["traversal_params"]["n_neighbors"]:
        :].tolist()

        top_neighbors = [neighbors[i] for i in top_neighbors_indices]
        return top_neighbors

    @staticmethod
    def get_prompt(prompt, curr_path, G):
        return "Question: " + prompt + " Evidence: " + " ".join(
            [G.nodes[node]["passage"] for node in curr_path]
        )

    @staticmethod
    def init_qwen_inference(args):
        model_name = args["retriever"]["model"]
        adapter_name = args["retriever"]["adapter"]
        load_in_4bit = args["retriever"].get("load_in_4bit", True) and torch.cuda.is_available()

        quant_config = None
        if load_in_4bit:
            quant_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
            )

        try:
            tokenizer = AutoTokenizer.from_pretrained(
                model_name,
                use_fast=True,
                trust_remote_code=True,
            )
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token

            base_model = AutoModelForCausalLM.from_pretrained(
                model_name,
                trust_remote_code=True,
                device_map="auto" if torch.cuda.is_available() else None,
                torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
                quantization_config=quant_config,
                low_cpu_mem_usage=True,
            )

            model = PeftModel.from_pretrained(base_model, adapter_name, is_trainable=False)
            print("Loaded Qwen in PEFT mode.")
        except Exception as e:
            print("PEFT load failed:", e)
            print("Falling back to direct model load...")

            tokenizer = AutoTokenizer.from_pretrained(
                adapter_name,
                use_fast=True,
                trust_remote_code=True,
            )
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token

            model = AutoModelForCausalLM.from_pretrained(
                adapter_name,
                trust_remote_code=True,
                device_map="auto" if torch.cuda.is_available() else None,
                torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
                quantization_config=quant_config,
                low_cpu_mem_usage=True,
            )

        model.eval()

        temp = args["retriever"]["inference_params"]["temp"]
        top_p = args["retriever"]["inference_params"]["top_p"]
        max_token_len = args["retriever"]["inference_params"]["max_token_len"]

        return Qwen_Inference(
            model=model,
            tokenizer=tokenizer,
            temp=temp if temp else 1.0,
            top_p=top_p if top_p else 1.0,
            max_token_len=max_token_len,
            parse_template=True,
        )

    @staticmethod
    def init_sentence_transformer(args):
        return SentenceTransformer(args["emb_model"]["model"])

In [ ]:
# Cell 10
# Load config, dataset, graph, and models
import os
import json
import yaml
import torch

args = yaml.safe_load(open(CFG_PATH, "r", encoding="utf-8"))

args["root_dir"] = os.path.abspath(args["root_dir"])
args["device"] = "cuda" if torch.cuda.is_available() else "cpu"

with open(os.path.join(args["root_dir"], args["question_dataset"]), "r", encoding="utf-8") as f:
    dataset = json.load(f)

G = load_graph(os.path.join(args["root_dir"], args["KG"]))

print("Dataset loaded:", len(dataset))
print("Graph nodes:", G.number_of_nodes())
print("Graph edges:", G.number_of_edges())
print("Device:", args["device"])

retriever_name = args["init_retriever"]["name"]

init_retriever = get_seeding_retriever_local(
    retriever_name,
    topk=args["init_retriever"]["topk"],
    G=G,
    dataset=dataset,
)

if not args["init_retriever"]["no_traversal"]:
    traversal_agent = Qwen_Agent(args)
    print("Qwen traversal agent initialized.")
else:
    traversal_agent = None
    print("Traversal disabled.")

Dataset loaded: 1000
Graph nodes: 441795
Graph edges: 6379408
Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Qwen traversal agent initialized.


In [ ]:
# Cell 11
# Optional smoke test
RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    sample_idx = 0
    sample = dataset[sample_idx]

    print("Question:", sample["question"])
    print("Type:", sample.get("type", "unknown"))

    evidence = get_supporting_evidence(
        sample["question"],
        G,
        init_retriever,
        traversal_agent,
        args,
    )

    evidence = evidence or []

    print("Num evidence items:", len(evidence))
    print("\nFirst evidence item:\n")
    print(evidence[0][:2000] if evidence else "No evidence")
else:
    print("Smoke test skipped for resume.")

Smoke test skipped for resume.


In [ ]:
# Cell 11A
# Prepare resume checkpoint and safe sync
import os
import json
import shutil
import time
import random
from google.colab import drive

LOCAL_RUN_DIR = "/content/kgp_qwen_agent_resume"
os.makedirs(LOCAL_RUN_DIR, exist_ok=True)

DRIVE_SAVE_DIR = os.path.join(
    args["root_dir"],
    args["checkpoint"]["save_dir"],
    args["checkpoint"]["id"],
)

DRIVE_CP_PATH = os.path.join(DRIVE_SAVE_DIR, "cp_evidence.json")
DRIVE_FINAL_PATH = os.path.join(DRIVE_SAVE_DIR, "evidence.json")

LOCAL_CP_PATH = os.path.join(LOCAL_RUN_DIR, "cp_evidence.json")
LOCAL_FINAL_PATH = os.path.join(LOCAL_RUN_DIR, "evidence.json")
LOCAL_CONFIG_PATH = os.path.join(LOCAL_RUN_DIR, "config.yml")

os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

def load_json_list(path):
    with open(path, "r", encoding="utf-8") as f:
        records = json.load(f)
    if not isinstance(records, list):
        raise ValueError(f"JSON is not a list: {path}")
    return records

def atomic_write_json_local(obj, path):
    tmp_path = path + ".tmp"
    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=4, ensure_ascii=False)
    os.replace(tmp_path, path)

def atomic_write_yaml_local(obj, path):
    tmp_path = path + ".tmp"
    with open(tmp_path, "w", encoding="utf-8") as f:
        yaml.dump(obj, f, sort_keys=False)
    os.replace(tmp_path, path)

def sync_file_to_drive(local_path, drive_path, retries=2):
    os.makedirs(os.path.dirname(drive_path), exist_ok=True)
    tmp_drive_path = drive_path + ".tmp"

    for attempt in range(retries + 1):
        try:
            shutil.copy2(local_path, tmp_drive_path)
            os.replace(tmp_drive_path, drive_path)
            return True
        except Exception as e:
            print(f"Drive sync failed attempt {attempt + 1}/{retries + 1}: {repr(e)}")
            try:
                if os.path.exists(tmp_drive_path):
                    os.remove(tmp_drive_path)
            except Exception:
                pass

            if attempt < retries:
                time.sleep(5)
                try:
                    drive.mount("/content/drive", force_remount=True)
                except Exception as mount_err:
                    print("Drive remount failed:", repr(mount_err))

    return False

def save_checkpoint(records, make_backup=False):
    atomic_write_json_local(records, LOCAL_CP_PATH)

    ok = sync_file_to_drive(LOCAL_CP_PATH, DRIVE_CP_PATH, retries=2)

    if ok and make_backup:
        backup_name = f"cp_evidence_backup_{len(records):04d}_{time.strftime('%Y%m%d_%H%M%S')}.json"
        backup_path = os.path.join(DRIVE_SAVE_DIR, backup_name)
        sync_file_to_drive(LOCAL_CP_PATH, backup_path, retries=1)

    return ok

def save_final(records):
    atomic_write_json_local(records, LOCAL_FINAL_PATH)
    ok = sync_file_to_drive(LOCAL_FINAL_PATH, DRIVE_FINAL_PATH, retries=3)
    return ok

# Load best checkpoint selected in Cell 6A
records = load_json_list(BEST_CP_SOURCE)

atomic_write_json_local(records, LOCAL_CP_PATH)
atomic_write_yaml_local(args, LOCAL_CONFIG_PATH)
sync_file_to_drive(LOCAL_CONFIG_PATH, os.path.join(DRIVE_SAVE_DIR, "config.yml"), retries=2)

processed_questions = set()
bad_records = 0

for r in records:
    q = r.get("question")
    if q:
        processed_questions.add(q)
    else:
        bad_records += 1

print("Loaded checkpoint records:", len(records))
print("Unique processed questions:", len(processed_questions))
print("Bad records:", bad_records)
print("Local checkpoint:", LOCAL_CP_PATH)
print("Drive checkpoint:", DRIVE_CP_PATH)

Loaded checkpoint records: 545
Unique processed questions: 545
Bad records: 0
Local checkpoint: /content/kgp_qwen_agent_resume/cp_evidence.json
Drive checkpoint: /content/drive/.shortcut-targets-by-id/1V7smEWLD_ZhlaD773UjRiHThZZ9cpgS-/final_project/baseline/DATA/KG/evidence/hotpot_evidence_1000/qwen_agent/cp_evidence.json


In [14]:
# cell 12
# Resume evidence generation from checkpoint
from tqdm import tqdm
import traceback

question_indices = list(range(len(dataset)))
random.seed(args["seed"])
random.shuffle(question_indices)

questions = [(idx, dataset[idx]) for idx in question_indices]

remaining_questions = [
    (idx, q)
    for idx, q in questions
    if q["question"] not in processed_questions
]

print("Total dataset questions:", len(dataset))
print("Checkpoint records:", len(records))
print("Remaining questions:", len(remaining_questions))

if len(remaining_questions) == 0:
    print("Nothing to do. Checkpoint already contains all questions.")

save_every = int(args["checkpoint"].get("save_every", 1))
backup_every = 25
consecutive_drive_failures = 0

for step, (idx, q) in enumerate(tqdm(remaining_questions, total=len(remaining_questions)), start=1):
    user_query = q["question"]

    if user_query in processed_questions:
        continue

    try:
        if retriever_name in ["gold", "none"]:
            evidence = init_retriever.retrieve(user_query)
        else:
            if args["init_retriever"]["no_traversal"]:
                init_retriever.topk = args["init_retriever"]["no_traversal_topk"]
                evidence_indices = init_retriever.retrieve(user_query)
                evidence = [init_retriever.all_documents[i] for i in evidence_indices]
            else:
                evidence = get_supporting_evidence(
                    user_query,
                    G,
                    init_retriever,
                    traversal_agent,
                    args,
                )

        if evidence or retriever_name == "none":
            q_type = q.get("type", "unknown")
            record = generate_evidence_record(
                q_type,
                user_query,
                evidence,
                q.get("answer", ""),
                q.get("supports", []),
            )
            records.append(record)
            processed_questions.add(user_query)
        else:
            print(f"No evidence found for question index {idx}: {user_query}")

        if step % save_every == 0:
            make_backup = (len(records) % backup_every == 0)
            ok = save_checkpoint(records, make_backup=make_backup)

            if ok:
                consecutive_drive_failures = 0
            else:
                consecutive_drive_failures += 1
                print(
                    "WARNING: Checkpoint saved locally but Drive sync failed. "
                    f"Local file: {LOCAL_CP_PATH}"
                )

    except Exception as e:
        print("Error while processing question index:", idx)
        print("Question:", user_query)
        print("Exception:", repr(e))
        traceback.print_exc()

        print("Saving checkpoint before stopping...")
        save_checkpoint(records, make_backup=True)
        raise

print("Loop finished. Saving final files...")

save_checkpoint(records, make_backup=True)
final_ok = save_final(records)

if not final_ok:
    print("WARNING: Final file saved locally but Drive final sync failed.")
    print("Local final file:", LOCAL_FINAL_PATH)
else:
    print("Final file synced to Drive.")

print("Finished.")
print("Drive save dir:", os.path.abspath(DRIVE_SAVE_DIR))
print("Drive checkpoint file:", os.path.abspath(DRIVE_CP_PATH))
print("Drive final evidence file:", os.path.abspath(DRIVE_FINAL_PATH))
print("Local checkpoint file:", os.path.abspath(LOCAL_CP_PATH))
print("Local final evidence file:", os.path.abspath(LOCAL_FINAL_PATH))
print("Total records:", len(records))

if len(records) != len(dataset):
    print("WARNING: Total records is not equal to dataset size.")
else:
    print("All dataset questions are completed.")

Total dataset questions: 1000
Checkpoint records: 545
Remaining questions: 455



100%|██████████| 455/455 [9:52:57<00:00, 78.19s/it]


Loop finished. Saving final files...
Final file synced to Drive.
Finished.
Drive save dir: /content/drive/.shortcut-targets-by-id/1V7smEWLD_ZhlaD773UjRiHThZZ9cpgS-/final_project/baseline/DATA/KG/evidence/hotpot_evidence_1000/qwen_agent
Drive checkpoint file: /content/drive/.shortcut-targets-by-id/1V7smEWLD_ZhlaD773UjRiHThZZ9cpgS-/final_project/baseline/DATA/KG/evidence/hotpot_evidence_1000/qwen_agent/cp_evidence.json
Drive final evidence file: /content/drive/.shortcut-targets-by-id/1V7smEWLD_ZhlaD773UjRiHThZZ9cpgS-/final_project/baseline/DATA/KG/evidence/hotpot_evidence_1000/qwen_agent/evidence.json
Local checkpoint file: /content/kgp_qwen_agent_resume/cp_evidence.json
Local final evidence file: /content/kgp_qwen_agent_resume/evidence.json
Total records: 1000
All dataset questions are completed.


In [15]:
# cell 13
# Verify saved output
import os
import json
from collections import Counter

save_path = os.path.join(args["root_dir"], args["checkpoint"]["save_dir"], args["checkpoint"]["id"])
final_file = os.path.join(save_path, "evidence.json")
config_file = os.path.join(save_path, "config.yml")

print("save_path:", os.path.abspath(save_path))
print("evidence.json exists:", os.path.isfile(final_file))
print("config.yml exists:", os.path.isfile(config_file))

records = json.load(open(final_file, "r", encoding="utf-8"))
print("num_records:", len(records))
print("type_counts:", Counter(r.get("type", "unknown") for r in records))

print("\nFirst saved record:")
print(json.dumps(records[0], ensure_ascii=False, indent=2)[:4000])

save_path: /content/drive/.shortcut-targets-by-id/1V7smEWLD_ZhlaD773UjRiHThZZ9cpgS-/final_project/baseline/DATA/KG/evidence/hotpot_evidence_1000/qwen_agent
evidence.json exists: True
config.yml exists: True
num_records: 1000
type_counts: Counter({'bridge': 700, 'comparison': 300})

First saved record:
{
  "type": "comparison",
  "question": "Where operation Operation Dragoon and Battle of Cold Harbor fought during to different wars?",
  "evidence": [
    "Title: Operation Dragoon. Evidence: Operation Dragoon also had political implications.",
    "Title: Operation Dragoon. Evidence: Despite these successes, there was criticism of Dragoon by some Allied generals and contemporary commentators such as Bernard Montgomery, Arthur R. Wilson, and Chester Wilmot in the aftermath, mostly because of its geo-strategic implications.",
    "Title: Operation Dragoon. Evidence: In the northeast the German problems loomed as large.",
    "Title: Operation Dragoon. Evidence: Dragoon therefore had conse